## **05. Set E — ABSA BERT Sentiment Scoring**

This notebook applies `yangheng/deberta-v3-base-absa-v1.1` to extract aspect-level sentiment scores for five service dimensions:

| Aspect | Label passed to model |
|---|---|
| seat | `seat comfort and legroom` |
| food | `food and beverage quality` |
| staff | `cabin crew and customer service` |
| ground_service | `flight delay, baggage, and check-in` |
| entertainment | `inflight entertainment and wifi` |

**Output:** `05_absa_scores.csv` — original dataframe + 5 new `absa_*` columns (weighted score: P_pos − P_neg, range −1 to +1)

> **Note:** This model was trained on restaurant/laptop/retail reviews (SemEval-2014, MAMS, etc.), not airline reviews. Domain mismatch is an acknowledged limitation, particularly for `ground_service`. See final report for discussion.

> ### **Rationale for Set E1 and Set E2**
>
> The ABSA-BERT model (`yangheng/deberta-v3-base-absa-v1.1`) is designed to return a sentiment score for a given aspect regardless of whether that aspect is explicitly mentioned in the review text. When an aspect is not mentioned, the model infers a score based on the overall context of the review — a behaviour that is fundamentally different from the rule-based approach in Set C, where unmentioned aspects return `NaN`.
>
> This distinction motivates the construction of two BERT-based feature sets:
>
> - **Set E1 (Full Inference):** ABSA-BERT scores are retained for all five aspects across all reviews, including cases where the aspect is not explicitly mentioned. This allows the model to leverage implicit sentiment signals embedded in the overall review context.
>
> - **Set E2 (Keyword-Gated Inference):** BERT inference is only executed for aspects that are explicitly mentioned in the review, as determined by the same keyword dictionary used in Set C. Unmentioned aspects are assigned `NaN`, consistent with Set C's treatment. This ensures that the only difference between Set C and Set E2 is the scoring method (VADER vs BERT), enabling a direct and fair comparison of the two approaches under identical mention coverage.
>
> This two-version design enables the following comparisons in the ablation study:
>
> | Comparison | What it reveals |
> |---|---|
> | Set C vs Set E2 | Effect of scoring method (VADER vs BERT) under identical mention coverage |
> | Set C vs Set E1 | Effect of expanding coverage to implicit sentiment via BERT inference |
> | Set E1 vs Set E2 | Whether BERT's contextual inference for unmentioned aspects adds or detracts predictive value |
>
> The sanity checks conducted prior to full inference confirmed that Set E1 scores for unmentioned aspects tend to reflect overall review sentiment rather than aspect-specific signals, which may introduce noise into the classifier. Set E2 is designed to mitigate this risk while ensuring methodological consistency with Set C.


## **1. Environment Setup**

In [1]:
# Install dependencies (Colab)
!pip install transformers torch -q

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


## **2. Mount Google Drive & Load Data**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# ── Update this path to match your Google Drive structure ──
DATA_PATH = '/content/drive/MyDrive/Airline-Review-Sentiment-Classifier/1_data/processed/04_aspect_scores.csv'
OUT_PATH  = '/content/drive/MyDrive/Airline-Review-Sentiment-Classifier/1_data/processed/05_absa_bert_scores.csv'

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape}")
df.head(3)

Loaded: (22980, 26)


,Airline Name,Verified,Type Of Traveller,Seat Type,Seat Comfort,Cabin Staff Service,Food & Beverages,Ground Service,Inflight Entertainment,Wifi & Connectivity,...,cleaned_review_C,vader_compound,vader_pos,vader_neg,vader_neu,aspect_seat,aspect_food,aspect_staff,aspect_ground_service,aspect_entertainment
0,AB Aviation,True,Solo Leisure,Economy Class,4.0,5.0,4.0,4.0,NaN,NaN,...,pretty decent airline moroni moheli turned pre...,0.9342,0.217,0.000,0.783,NaN,NaN,NaN,0.9468,NaN
1,AB Aviation,True,Solo Leisure,Economy Class,2.0,2.0,1.0,1.0,NaN,NaN,...,good airline moroni anjouan small airline tick...,-0.8244,0.027,0.091,0.882,NaN,NaN,NaN,0.4215,NaN
2,AB Aviation,True,Solo Leisure,Economy Class,2.0,1.0,1.0,1.0,NaN,NaN,...,flight fortunately short anjouan dzaoudzi smal...,0.7569,0.106,0.027,0.867,NaN,NaN,0.8122,NaN,0.8122


## **3. Load ABSA Model**

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_NAME = 'yangheng/deberta-v3-base-absa-v1.1'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# top_k=None returns all class probabilities (Negative / Neutral / Positive)
absa_classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    top_k=None,
    device=0 if torch.cuda.is_available() else -1
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/372 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Model loaded.


## **4. Define Aspect Labels**

In [6]:
# Aspect labels passed to the model as text_pair
# Expanded from single-word keys to fuller phrases to mitigate domain mismatch
ASPECT_LABELS = {
    'seat'            : 'seat comfort and legroom',
    'food'            : 'food and beverage quality',
    'staff'           : 'cabin crew and customer service',
    'ground_service'  : 'flight delay, baggage, and check-in',
    'entertainment'   : 'inflight entertainment and wifi'
}

## **5. Define Scoring Function**

Model output per aspect: `[{label: 'Negative', score: P_neg}, {label: 'Neutral', score: P_neu}, {label: 'Positive', score: P_pos}]`

Weighted score = `P_pos − P_neg` → range −1 to +1, directly comparable to Set C VADER compound scores.

In [10]:
def get_weighted_score(probs: list) -> float:
    """Convert probability list to weighted sentiment score (P_pos - P_neg)."""
    prob_dict = {item['label']: item['score'] for item in probs}
    return round(prob_dict.get('Positive', 0) - prob_dict.get('Negative', 0), 4)


def score_aspects(text: str) -> dict:
    """Return weighted ABSA score for each aspect given a review text."""
    if not isinstance(text, str) or text.strip() == '':
        return {f'absa_{asp}': None for asp in ASPECT_LABELS}

    # Truncate to 512 tokens (DeBERTa limit) — pipeline handles this but explicit is safer
    text = text[:2000]

    results = {}
    for asp, label in ASPECT_LABELS.items():
        try:
            probs = absa_classifier(text, text_pair=label)[0]
            results[f'absa_{asp}'] = get_weighted_score(probs)
        except Exception as e:
            results[f'absa_{asp}'] = None
    return results


# ── Quick sanity check ──
sample = df['cleaned_review_BE'].iloc[0]
print("Sample review:", sample)
print("\nABSA scores:")
print(score_aspects(sample))

Sample review: pretty decent airline. moroni to moheli. turned out to be a pretty decent airline. online booking worked well, checkin and boarding was fine and the plane looked well maintained. its a very short flight just 20 minutes or so so i did not expect much but they still managed to hand our a bottle of water and some biscuits which i though was very nice. both flights on time.

ABSA scores:
{'absa_seat': 0.9722, 'absa_food': 0.9598, 'absa_staff': 0.9679, 'absa_ground_service': 0.9611, 'absa_entertainment': 0.9552}


In [9]:
test = "The seat was extremely uncomfortable and cramped. However, the cabin crew were absolutely wonderful and very attentive."
print(score_aspects(test))

{'absa_seat': -0.9871, 'absa_food': -0.067, 'absa_staff': 0.9936, 'absa_ground_service': -0.0202, 'absa_entertainment': 0.2754}


In [12]:
# 테스트 1 — entertainment만 부정, 나머지 언급 없음
test1 = "The screen was broken and there was no wifi available on this long flight."
print("Test 1:", score_aspects(test1))

# 테스트 2 — 음식만 부정, 나머지 긍정
test2 = "Crew was fantastic and seat was very comfortable. But the food was absolutely terrible, cold and tasteless."
print("Test 2:", score_aspects(test2))

# 테스트 3 — ground service만 부정
test3 = "Flight was delayed 4 hours with no explanation. Luggage was lost and never recovered."
print("Test 3:", score_aspects(test3))

Test 1: {'absa_seat': 0.9443, 'absa_food': -0.6027, 'absa_staff': 0.2956, 'absa_ground_service': -0.0532, 'absa_entertainment': -0.9848}
Test 2: {'absa_seat': 0.9967, 'absa_food': -0.9305, 'absa_staff': 0.9625, 'absa_ground_service': -0.2241, 'absa_entertainment': 0.765}
Test 3: {'absa_seat': 0.9382, 'absa_food': -0.8859, 'absa_staff': -0.9478, 'absa_ground_service': -0.9914, 'absa_entertainment': -0.0149}


## **6. Run Inference on Full Dataset**

> ⏱️ Expected time: ~30–60 min for 22,980 rows on Colab T4 GPU.  
> Progress is saved to Drive every 1,000 rows so you can resume if the session drops.

In [8]:
import os
from tqdm.auto import tqdm

CHECKPOINT_PATH = OUT_PATH.replace('.csv', '_checkpoint.csv')
SAVE_EVERY = 1000

# ── Resume from checkpoint if it exists ──
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(df_done)
    print(f"Resuming from row {start_idx}")
else:
    df_done = df.copy()
    for col in [f'absa_{asp}' for asp in ASPECT_LABELS]:
        df_done[col] = None
    start_idx = 0
    print("Starting from scratch.")

# ── Inference loop ──
texts = df['cleaned_review_BE'].tolist()

for i in tqdm(range(start_idx, len(df)), desc='ABSA scoring'):
    scores = score_aspects(texts[i])
    for col, val in scores.items():
        df_done.at[i, col] = val

    if (i + 1) % SAVE_EVERY == 0:
        df_done.to_csv(CHECKPOINT_PATH, index=False)
        print(f"  Checkpoint saved at row {i+1}")

print("\nInference complete.")

Starting from scratch.


ABSA scoring:   0%|          | 0/22980 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


KeyboardInterrupt: 

## **7. Save Final Output**

In [ ]:
df_done.to_csv(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH}")
print(f"Final shape: {df_done.shape}")
df_done.info()

## **8. Quick Validation**

In [ ]:
absa_cols = [f'absa_{asp}' for asp in ASPECT_LABELS]

print("=== Score Distribution ===")
print(df_done[absa_cols].describe().round(3))

print("\n=== Non-null counts ===")
print(df_done[absa_cols].notna().sum())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for ax, col in zip(axes, absa_cols):
    df_done[col].dropna().hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col.replace('absa_', ''))
    ax.set_xlabel('Score (P_pos - P_neg)')
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
plt.suptitle('Set E — ABSA Score Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Set C vs Set E on same aspects (correlation check)
compare_pairs = [
    ('aspect_seat',           'absa_seat'),
    ('aspect_food',           'absa_food'),
    ('aspect_staff',          'absa_staff'),
    ('aspect_ground_service', 'absa_ground_service'),
    ('aspect_entertainment',  'absa_entertainment'),
]

print("=== Pearson Correlation: Set C vs Set E ===")
for c_col, e_col in compare_pairs:
    corr = df_done[[c_col, e_col]].dropna().corr().iloc[0, 1]
    print(f"  {c_col:<30} vs {e_col:<25} r = {corr:.3f}")

# **E2 - with Masking**

In [15]:
!pip install vaderSentiment -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.1 MB/s eta 0:00:00


In [18]:
# Set E2 Masking Sanity Check
test_reviews = [
    "The screen was broken and there was no wifi available on this long flight.",
    "Crew was fantastic and seat was very comfortable. But the food was absolutely terrible, cold and tasteless.",
    "Flight was delayed 4 hours with no explanation. Luggage was lost and never recovered."
]

import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
from nltk.tokenize import sent_tokenize
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

analyzer_test = SentimentIntensityAnalyzer()

aspect_keywords = {
    'seat': ['seat', 'legroom', 'comfort', 'recline', 'space', 'cushion', 'comfy', 'spacious', 'stretch'],
    'food': ['food', 'meal', 'drink', 'snack', 'beverage', 'dining', 'menu', 'vegetarian'],
    'staff': ['crew', 'staff', 'attendant', 'stewardess', 'service', 'friendly', 'rude', 'attentive'],
    'ground_service': ['delay', 'late', 'on time', 'punctual', 'schedule', 'depart', 'cancel', 'luggage', 'suitcase',
                       'baggage', 'lost', 'checkin', 'check-in', 'refund', 'booking', ' boarding'],
    'entertainment': ['entertainment', 'screen', 'movie', 'ife', 'wifi', 'music']  # ife stands for In-Flight Entertainment
}

def score_aspects_e2(text):
    """
    Run BERT inference only for aspects explicitly mentioned in the review,
    using the same keyword rules applied in Set C.
    Returns None for unmentioned aspects.
    """
    if not isinstance(text, str) or text.strip() == '':
        return {f'absa_{asp}_e2': None for asp in aspect_keywords}

    text = text[:2000]
    results = {}

    for asp, keywords in aspect_keywords.items():
        if has_aspect_mention(text, keywords):
            try:
                probs = absa_classifier(text, text_pair=ASPECT_LABELS[asp])[0]
                results[f'absa_{asp}_e2'] = get_weighted_score(probs)
            except Exception:
                results[f'absa_{asp}_e2'] = None
        else:
            results[f'absa_{asp}_e2'] = None

    return results


# ── Sanity check ──
print("=== E1 vs E2 Sanity Check ===\n")
test_reviews = [
    "The screen was broken and there was no wifi available on this long flight.",
    "Crew was fantastic and seat was very comfortable. But the food was absolutely terrible, cold and tasteless.",
    "Flight was delayed 4 hours with no explanation. Luggage was lost and never recovered."
]

for i, review in enumerate(test_reviews, 1):
    print(f"Test {i}: {review[:80]}...")
    e1 = score_aspects(review)
    e2 = score_aspects_e2(review)
    print(f"  E1: {e1}")
    print(f"  E2: {e2}")
    print()

=== E1 vs E2 Sanity Check ===

Test 1: The screen was broken and there was no wifi available on this long flight....
  E1: {'absa_seat': 0.9443, 'absa_food': -0.6027, 'absa_staff': 0.2956, 'absa_ground_service': -0.0532, 'absa_entertainment': -0.9848}
  E2: {'absa_seat_e2': None, 'absa_food_e2': None, 'absa_staff_e2': None, 'absa_ground_service_e2': None, 'absa_entertainment_e2': -0.9848}

Test 2: Crew was fantastic and seat was very comfortable. But the food was absolutely te...
  E1: {'absa_seat': 0.9967, 'absa_food': -0.9305, 'absa_staff': 0.9625, 'absa_ground_service': -0.2241, 'absa_entertainment': 0.765}
  E2: {'absa_seat_e2': 0.9967, 'absa_food_e2': -0.9305, 'absa_staff_e2': 0.9625, 'absa_ground_service_e2': None, 'absa_entertainment_e2': None}

Test 3: Flight was delayed 4 hours with no explanation. Luggage was lost and never recov...
  E1: {'absa_seat': 0.9382, 'absa_food': -0.8859, 'absa_staff': -0.9478, 'absa_ground_service': -0.9914, 'absa_entertainment': -0.0149}
  E2: {'a

In [ ]:
import os
from tqdm.auto import tqdm

CHECKPOINT_PATH = OUT_PATH.replace('.csv', '_checkpoint.csv')
SAVE_EVERY = 1000

absa_cols = [f'absa_{asp}' for asp in ASPECT_LABELS]

# ── Resume from checkpoint if exists ──
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_csv(CHECKPOINT_PATH)
    start_idx = len(df_done)
    print(f"Resuming from row {start_idx}")
else:
    df_done = df.copy()
    for col in absa_cols:
        df_done[col] = None
    start_idx = 0
    print("Starting from scratch.")

# ── Full Inference (Set E1) ──
texts = df['cleaned_review_BE'].tolist()

for i in tqdm(range(start_idx, len(df)), desc='ABSA scoring (E1)'):
    scores = score_aspects(texts[i])
    for col, val in scores.items():
        df_done.at[i, col] = val

    if (i + 1) % SAVE_EVERY == 0:
        df_done.to_csv(CHECKPOINT_PATH, index=False)
        print(f"  Checkpoint saved at row {i+1}")

print("\nE1 inference complete.")

# ── Set E2: Apply Set C Masking ──
e2_cols = [f'absa_{asp}_e2' for asp in aspect_keywords]
for col in e2_cols:
    df_done[col] = None

for asp, keywords in aspect_keywords.items():
    texts_list = df['cleaned_review_BE'].tolist()
    for i in range(len(df_done)):
        if has_aspect_mention(texts_list[i], keywords):
            df_done.at[i, f'absa_{asp}_e2'] = df_done.at[i, f'absa_{asp}']
        # else: None 유지

print("E2 masking applied.")

# ── Save Final Output ──
df_done.to_csv(OUT_PATH, index=False)
print(f"\nSaved → {OUT_PATH}")
print(f"Final shape: {df_done.shape}")
df_done.info()